# Capítulo 9: Prompts Complexos do Zero

- [Lição](#lesson)
- [Exercícios](#exercises)
- [Playground de Exemplos](#example-playground)

## Configuração

Execute a célula de configuração abaixo para carregar sua chave de API e estabelecer a função auxiliar `get_completion`.

In [ ]:
import sys, re
sys.path.insert(0, '/workspaces/prompt-eng-ia')
from ollama_helper import get_completion, MODEL_NAME, client

---

## Lição

Parabéns por chegar ao último capítulo! Agora é hora de juntar tudo e aprender a **criar prompts únicos e complexos**. 

Abaixo, você usará uma **estrutura guiada que recomendamos para prompts complexos**. Nas partes posteriores deste capítulo, mostraremos alguns prompts específicos de setores e explicaremos como esses prompts são estruturados de forma similar.

**Nota:** **Nem todos os prompts precisam de todos os elementos da estrutura complexa a seguir**. Incentivamos você a experimentar e incluir ou excluir elementos para ver como isso afeta a resposta do Claude. Geralmente é **melhor usar muitos elementos de prompt para fazer seu prompt funcionar primeiro e, em seguida, refinar e simplificar seu prompt depois**.

### Exemplo - Chatbot de Coach de Carreira

A estrutura a seguir combina múltiplos elementos de engenharia de prompt e é um bom ponto de partida para prompts complexos. **A ordem importa para alguns elementos**, não para outros. Anotaremos quando as melhores práticas indicam que a ordem importa, mas em geral, **se você seguir esta ordem, será um ótimo começo para um prompt excepcional**.

No exemplo a seguir, construiremos um prompt para um roleplay controlado no qual o Claude assume um papel situacional com uma tarefa específica. Nosso objetivo é fazer o Claude agir como um coach de carreira amigável.

Leia e execute a célula abaixo para compilar os vários elementos do prompt em um prompt completo.

In [ ]:
######################################## VARIÁVEIS DE ENTRADA ########################################

# Primeira variável de entrada - o histórico da conversa (também pode ser adicionado como mensagens anteriores `user` e `assistant` na chamada da API)
HISTORY = """Customer: Give me two possible careers for sociology majors.

Joe: Here are two potential careers for sociology majors:

- Social worker - Sociology provides a strong foundation for understanding human behavior and social systems. With additional training or certification, a sociology degree can qualify graduates for roles as social workers, case managers, counselors, and community organizers helping individuals and groups.

- Human resources specialist - An understanding of group dynamics and organizational behavior from sociology is applicable to careers in human resources. Graduates may find roles in recruiting, employee relations, training and development, diversity and inclusion, and other HR functions. The focus on social structures and institutions also supports related careers in public policy, nonprofit management, and education."""

# Segunda variável de entrada - a pergunta do usuário
QUESTION = "Which of the two careers requires more than a Bachelor's degree?"



######################################## ELEMENTOS DO PROMPT ########################################

##### Elemento do prompt 1: role `user`
# Certifique-se de que sua chamada à API de Mensagens sempre comece com um role `user` no array de mensagens.
# A função get_completion() conforme definida acima fará isso automaticamente para você.

##### Elemento do prompt 2: Contexto da tarefa
# Forneça ao Claude contexto sobre o papel que ele deve assumir ou quais metas e tarefas gerais você quer que ele execute com o prompt.
# É melhor colocar o contexto no início do corpo do prompt.
TASK_CONTEXT = "You will be acting as an AI career coach named Joe created by the company AdAstra Careers. Your goal is to give career advice to users. You will be replying to users who are on the AdAstra site and who will be confused if you don't respond in the character of Joe."

##### Elemento do prompt 3: Contexto de tom
# Se importante para a interação, diga ao Claude qual tom ele deve usar.
# Este elemento pode não ser necessário dependendo da tarefa.
TONE_CONTEXT = "You should maintain a friendly customer service tone."

##### Elemento do prompt 4: Descrição detalhada da tarefa e regras
# Expanda sobre as tarefas específicas que você quer que o Claude faça, bem como quaisquer regras que o Claude possa ter que seguir.
# É aqui também que você pode dar ao Claude uma "saída" se ele não tiver uma resposta ou não souber.
# É ideal mostrar esta descrição e regras a um amigo para garantir que estejam dispostas de forma lógica e que quaisquer palavras ambíguas estejam claramente definidas.
TASK_DESCRIPTION = """Here are some important rules for the interaction:
- Always stay in character, as Joe, an AI from AdAstra Careers
- If you are unsure how to respond, say \"Sorry, I didn't understand that. Could you rephrase your question?\"
- If someone asks something irrelevant, say, \"Sorry, I am Joe and I give career advice. Do you have a career question today I can help you with?\""""

##### Elemento do prompt 5: Exemplos
# Forneça ao Claude pelo menos um exemplo de resposta ideal que ele possa emular. Coloque dentro de tags XML <example></example>. Fique à vontade para fornecer múltiplos exemplos.
# Se você fornecer múltiplos exemplos, dê ao Claude contexto sobre o que é um exemplo, e coloque cada exemplo em seu próprio conjunto de tags XML.
# Exemplos são provavelmente a ferramenta mais eficaz em trabalho de conhecimento para fazer o Claude se comportar como desejado.
# Certifique-se de dar ao Claude exemplos de casos extremos comuns. Se seu prompt usa um rascunho, é eficaz dar exemplos de como o rascunho deve parecer.
# Em geral, mais exemplos = melhor.
EXAMPLES = """Here is an example of how to respond in a standard interaction:
<example>
Customer: Hi, how were you created and what do you do?
Joe: Hello! My name is Joe, and I was created by AdAstra Careers to give career advice. What can I help you with today?
</example>"""

##### Elemento do prompt 6: Dados de entrada para processar
# Se houver dados que o Claude precise processar no prompt, inclua-os aqui dentro de tags XML relevantes.
# Fique à vontade para incluir múltiplos dados, mas certifique-se de colocar cada um em seu próprio conjunto de tags XML.
# Este elemento pode não ser necessário dependendo da tarefa. A ordem também é flexível.
INPUT_DATA = f"""Here is the conversational history (between the user and you) prior to the question. It could be empty if there is no history:
<history>
{HISTORY}
</history>

Here is the user's question:
<question>
{QUESTION}
</question>"""

##### Elemento do prompt 7: Descrição ou solicitação imediata da tarefa #####
# "Lembre" ao Claude ou diga ao Claude exatamente o que ele deve fazer imediatamente para cumprir a tarefa do prompt.
# É aqui também que você colocaria variáveis adicionais como a pergunta do usuário.
# Geralmente não há problema em reiterar ao Claude sua tarefa imediata. É melhor fazer isso no final de um prompt longo.
# Isso produzirá melhores resultados do que colocar isso no início.
# Também é geralmente boa prática colocar a consulta do usuário próxima ao final do prompt.
IMMEDIATE_TASK = "How do you respond to the user's question?"

##### Elemento do prompt 8: Pré-cognição (pensar passo a passo)
# Para tarefas com múltiplas etapas, é bom dizer ao Claude para pensar passo a passo antes de dar uma resposta
# Às vezes, você pode até ter que dizer "Antes de dar sua resposta..." apenas para garantir que o Claude faça isso primeiro.
# Não é necessário em todos os prompts, mas se incluído, é melhor fazer isso no final de um prompt longo e logo após a solicitação ou descrição final da tarefa imediata.
PRECOGNITION = "Think about your answer first before you respond."

##### Elemento do prompt 9: Formatação de saída
# Se houver uma maneira específica de formatar a resposta do Claude, diga claramente ao Claude qual é esse formato.
# Este elemento pode não ser necessário dependendo da tarefa.
# Se você incluí-lo, colocá-lo no final do prompt é melhor do que no início.
OUTPUT_FORMATTING = "Put your response in <response></response> tags."

##### Elemento do prompt 10: Preenchimento da resposta do Claude (se houver)
# Um espaço para iniciar a resposta do Claude com algumas palavras pré-preenchidas para orientar o comportamento ou a resposta do Claude.
# Se você quiser pré-preencher a resposta do Claude, deve colocar isso no role `assistant` na chamada da API.
# Este elemento pode não ser necessário dependendo da tarefa.
PREFILL = "[Joe] <response>"



######################################## COMBINAR ELEMENTOS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Imprime o prompt completo
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print("TURNO DO USUÁRIO")
print(PROMPT)
print("\nTURNO DO ASSISTENTE")
print(PREFILL)

Agora vamos executar o prompt! Execute a célula abaixo para ver a saída do Claude.

In [ ]:
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

### Exemplo - Serviços Jurídicos

**Prompts na área jurídica podem ser bastante complexos** devido à necessidade de:
- Analisar documentos longos
- Lidar com tópicos complexos
- Formatar a saída de formas muito específicas
- Seguir processos analíticos em múltiplas etapas

Vamos ver como podemos usar o template de prompt complexo para estruturar um prompt para um caso de uso jurídico específico. Abaixo, detalhamos um exemplo de prompt para um caso de uso jurídico no qual pedimos ao Claude para responder perguntas sobre uma questão legal usando informações de um documento legal.

**Mudamos a ordem de alguns elementos** para mostrar que a estrutura do prompt pode ser flexível!

**Engenharia de prompt é sobre tentativa e erro científico**. Incentivamos você a misturar e combinar, mover as coisas (os elementos onde a ordem não importa) e ver o que funciona melhor para você e suas necessidades.

In [ ]:
######################################## VARIÁVEIS DE ENTRADA ########################################

# Primeira variável de entrada - o documento legal
LEGAL_RESEARCH = """<search_results>
<search_result id=1>
The animal health industry became caught up in a number of patent and trademark lawsuits during the past year. In 1994, Barclay Slocum obtained patents for the tibial plateau leveling osteotomy procedure, which is used in the treatment of dogs with cranial cruciate ligament rupture, and for the devices used in the procedure. During 2006, Slocum Enterprises filed a patent infringement suit against New Generation Devices, arguing that the Unity Cruciate Plate manufactured by New Generation infringed on the patent for the Slocum TPLO plate. However, the court never reached a decision on the issue of patent infringement, ruling that it did not have jurisdiction on the basis of the small number of plates sold in the state in which the case was filed and the information provided on a Web site maintained by Slocum Enterprises. Other patent battles waged during 2006 concerned the use of laser technology for onychectomy in cats, pet identification chips, pig vaccines, and pet “deshedding” tools.
</search_result>
<search_result id=2>
In Canada, the British Columbia Veterinary Medical Association brought suit against a nonveterinarian, claiming that he engaged in cutting or otherwise removing hooks from horses' teeth and floating horses' teeth with power and manual tools, provided advice and diagnoses in return for a fee, and held himself out as being qualified and willing to provide treatment with respect to these activities. The court held that the intention of the legislature in passing the Veterinary Profession Act was the protection of the public and animals and further held that monopolistic statutes serve the purpose of protecting the public. In addition, the court concluded that dentistry, at its core, relates to the health of the teeth and gums; is distinct from cosmetic and other types of care of animals; and, therefore, falls under the definition of the practice of veterinary medicine. The nonveterinarian was enjoined from providing services without a veterinarian supervising the procedures.
</search_result>
<search_result id=3>
The aftermath of Hurricane Katrina, which hit the Gulf Coast of the United States during 2005, spurred changes to the way animals are treated during natural disasters. In 2006, Hawaii, Louisiana, and New Hampshire all enacted laws that address issues regarding the care of animals during disasters, such as providing shelters for pets and allowing service animals to be kept with the people they serve. In addition, Congress passed, and the President signed, the Pet Evacuation and Transportation Standards Act during 2006, which requires state and local emergency preparedness authorities to include in their evacuation plans information on how they will accommodate household pets and service animals in case of a disaster. California passed a law that will require its Office of Emergency Services, Department of Agriculture, and other agencies involved with disaster response preparation to develop a plan for the needs of service animals, livestock, equids, and household pets in the event of a disaster or major emergency.
</search_result>
</search_results>"""

# Segunda variável de entrada - a pergunta do usuário
QUESTION = "Are there any laws about what to do with pets during a hurricane?"



######################################## ELEMENTOS DO PROMPT ########################################

##### Elemento do prompt 1: role `user`
# Certifique-se de que sua chamada à API de Mensagens sempre comece com um role `user` no array de mensagens.
# A função get_completion() conforme definida acima fará isso automaticamente para você.

##### Elemento do prompt 2: Contexto da tarefa
# Forneça ao Claude contexto sobre o papel que ele deve assumir ou quais metas e tarefas gerais você quer que ele execute com o prompt.
# É melhor colocar o contexto no início do corpo do prompt.
TASK_CONTEXT = "You are an expert lawyer."

##### Elemento do prompt 3: Contexto de tom
# Se importante para a interação, diga ao Claude qual tom ele deve usar.
# Este elemento pode não ser necessário dependendo da tarefa.
TONE_CONTEXT = ""

##### Elemento do prompt 4: Dados de entrada para processar
# Se houver dados que o Claude precise processar no prompt, inclua-os aqui dentro de tags XML relevantes.
# Fique à vontade para incluir múltiplos dados, mas certifique-se de colocar cada um em seu próprio conjunto de tags XML.
# Este elemento pode não ser necessário dependendo da tarefa. A ordem também é flexível.
INPUT_DATA = f"""Here is some research that's been compiled. Use it to answer a legal question from the user.
<legal_research>
{LEGAL_RESEARCH}
</legal_research>"""

##### Elemento do prompt 5: Exemplos
# Forneça ao Claude pelo menos um exemplo de resposta ideal que ele possa emular. Coloque dentro de tags XML <example></example>. Fique à vontade para fornecer múltiplos exemplos.
# Se você fornecer múltiplos exemplos, dê ao Claude contexto sobre o que é um exemplo, e coloque cada exemplo em seu próprio conjunto de tags XML.
# Exemplos são provavelmente a ferramenta mais eficaz em trabalho de conhecimento para fazer o Claude se comportar como desejado.
# Certifique-se de dar ao Claude exemplos de casos extremos comuns. Se seu prompt usa um rascunho, é eficaz dar exemplos de como o rascunho deve parecer.
# Em geral, mais exemplos = melhor.
EXAMPLES = """When citing the legal research in your answer, please use brackets containing the search index ID, followed by a period. Put these at the end of the sentence that's doing the citing. Examples of proper citation format:

<examples>
<example>
The statute of limitations expires after 10 years for crimes like this. [3].
</example>
<example>
However, the protection does not apply when it has been specifically waived by both parties. [5].
</example>
</examples>"""

##### Elemento do prompt 6: Descrição detalhada da tarefa e regras
# Expanda sobre as tarefas específicas que você quer que o Claude faça, bem como quaisquer regras que o Claude possa ter que seguir.
# É aqui também que você pode dar ao Claude uma "saída" se ele não tiver uma resposta ou não souber.
# É ideal mostrar esta descrição e regras a um amigo para garantir que estejam dispostas de forma lógica e que quaisquer palavras ambíguas estejam claramente definidas.
TASK_DESCRIPTION = """Write a clear, concise answer to this question:

<question>
{QUESTION}
</question>

It should be no more than a couple of paragraphs. If possible, it should conclude with a single sentence directly answering the user's question. However, if there is not sufficient information in the compiled research to produce such an answer, you may demur and write \"Sorry, I do not have sufficient information at hand to answer this question.\"."""

##### Elemento do prompt 7: Descrição ou solicitação imediata da tarefa #####
# "Lembre" ao Claude ou diga ao Claude exatamente o que ele deve fazer imediatamente para cumprir a tarefa do prompt.
# É aqui também que você colocaria variáveis adicionais como a pergunta do usuário.
# Geralmente não há problema em reiterar ao Claude sua tarefa imediata. É melhor fazer isso no final de um prompt longo.
# Isso produzirá melhores resultados do que colocar isso no início.
# Também é geralmente boa prática colocar a consulta do usuário próxima ao final do prompt.
IMMEDIATE_TASK = ""

##### Elemento do prompt 8: Pré-cognição (pensar passo a passo)
# Para tarefas com múltiplas etapas, é bom dizer ao Claude para pensar passo a passo antes de dar uma resposta
# Às vezes, você pode até ter que dizer "Antes de dar sua resposta..." apenas para garantir que o Claude faça isso primeiro.
# Não é necessário em todos os prompts, mas se incluído, é melhor fazer isso no final de um prompt longo e logo após a solicitação ou descrição final da tarefa imediata.
PRECOGNITION = "Before you answer, pull out the most relevant quotes from the research in <relevant_quotes> tags."

##### Elemento do prompt 9: Formatação de saída
# Se houver uma maneira específica de formatar a resposta do Claude, diga claramente ao Claude qual é esse formato.
# Este elemento pode não ser necessário dependendo da tarefa.
# Se você incluí-lo, colocá-lo no final do prompt é melhor do que no início.
OUTPUT_FORMATTING = "Put your two-paragraph response in <answer> tags."

##### Elemento do prompt 10: Preenchimento da resposta do Claude (se houver)
# Um espaço para iniciar a resposta do Claude com algumas palavras pré-preenchidas para orientar o comportamento ou a resposta do Claude.
# Se você quiser pré-preencher a resposta do Claude, deve colocar isso no role `assistant` na chamada da API.
# Este elemento pode não ser necessário dependendo da tarefa.
PREFILL = "<relevant_quotes>"



######################################## COMBINAR ELEMENTOS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Imprime o prompt completo
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print("TURNO DO USUÁRIO")
print(PROMPT)
print("\nTURNO DO ASSISTENTE")
print(PREFILL)

Agora vamos executar o prompt! Execute a célula abaixo para ver a saída do Claude.

In [ ]:
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Se você quiser experimentar os prompts da lição sem alterar nenhum conteúdo acima, role até o final do notebook para visitar o [**Playground de Exemplos**](#example-playground).

---

## Exercícios
- [Exercício 9.1 - Chatbot de Serviços Financeiros](#exercise-91---financial-services-chatbot)
- [Exercício 9.2 - Codebot](#exercise-92---codebot)

### Exercício 9.1 - Chatbot de Serviços Financeiros
Prompts na área financeira também podem ser bastante complexos por razões semelhantes às dos prompts jurídicos. Aqui está um exercício para um caso de uso financeiro, no qual o Claude é usado para **analisar informações fiscais e responder perguntas**. Assim como no exemplo de serviços jurídicos, mudamos a ordem de alguns elementos, pois nosso prompt de solução faz mais sentido com um fluxo diferente (porém, outras estruturas também funcionariam).

Sugerimos que você leia o conteúdo das variáveis (neste caso, `{QUESTION}` e `{TAX_CODE}`) para entender com qual conteúdo o Claude deve trabalhar. Certifique-se de referenciar `{QUESTION}` e `{TAX_CODE}` diretamente no seu prompt em algum lugar (usando sintaxe f-string como nos outros exemplos) para que o conteúdo real da variável possa ser substituído.

Preencha os campos dos elementos do prompt com conteúdo que corresponda à descrição e aos exemplos que você viu nos exemplos anteriores de prompts complexos. Depois de preencher todos os elementos do prompt que deseja preencher, execute a célula para ver o prompt concatenado e a resposta do Claude.

Lembre-se de que a engenharia de prompt raramente é puramente formulaica, especialmente para prompts grandes e complexos! É importante desenvolver casos de teste e **tentar uma variedade de prompts e estruturas de prompt para ver o que funciona melhor para cada situação**. Note que se você *mudar* a ordem dos elementos do prompt, também deve lembrar de alterar a ordem da concatenação na seção `COMBINAR ELEMENTOS`.

In [ ]:
######################################## VARIÁVEIS DE ENTRADA ########################################

# Primeira variável de entrada - a pergunta do usuário
QUESTION = "How long do I have to make an 83b election?"

# Segunda variável de entrada - o documento do código tributário que o Claude usará para responder à pergunta do usuário
TAX_CODE = """
(a)General rule
If, in connection with the performance of services, property is transferred to any person other than the person for whom such services are performed, the excess of—
(1)the fair market value of such property (determined without regard to any restriction other than a restriction which by its terms will never lapse) at the first time the rights of the person having the beneficial interest in such property are transferable or are not subject to a substantial risk of forfeiture, whichever occurs earlier, over
(2)the amount (if any) paid for such property,
shall be included in the gross income of the person who performed such services in the first taxable year in which the rights of the person having the beneficial interest in such property are transferable or are not subject to a substantial risk of forfeiture, whichever is applicable. The preceding sentence shall not apply if such person sells or otherwise disposes of such property in an arm's length transaction before his rights in such property become transferable or not subject to a substantial risk of forfeiture.
(b)Election to include in gross income in year of transfer
(1)In general
Any person who performs services in connection with which property is transferred to any person may elect to include in his gross income for the taxable year in which such property is transferred, the excess of—
(A)the fair market value of such property at the time of transfer (determined without regard to any restriction other than a restriction which by its terms will never lapse), over
(B)the amount (if any) paid for such property.
If such election is made, subsection (a) shall not apply with respect to the transfer of such property, and if such property is subsequently forfeited, no deduction shall be allowed in respect of such forfeiture.
(2)Election
An election under paragraph (1) with respect to any transfer of property shall be made in such manner as the Secretary prescribes and shall be made not later than 30 days after the date of such transfer. Such election may not be revoked except with the consent of the Secretary.

(c)Special rules
For purposes of this section—
(1)Substantial risk of forfeiture
The rights of a person in property are subject to a substantial risk of forfeiture if such person's rights to full enjoyment of such property are conditioned upon the future performance of substantial services by any individual.

(2)Transferability of property
The rights of a person in property are transferable only if the rights in such property of any transferee are not subject to a substantial risk of forfeiture.

(3)Sales which may give rise to suit under section 16(b) of the Securities Exchange Act of 1934
So long as the sale of property at a profit could subject a person to suit under section 16(b) of the Securities Exchange Act of 1934, such person's rights in such property are—
(A)subject to a substantial risk of forfeiture, and
(B)not transferable.
(4)For purposes of determining an individual's basis in property transferred in connection with the performance of services, rules similar to the rules of section 72(w) shall apply.
(d)Certain restrictions which will never lapse
(1)Valuation
In the case of property subject to a restriction which by its terms will never lapse, and which allows the transferee to sell such property only at a price determined under a formula, the price so determined shall be deemed to be the fair market value of the property unless established to the contrary by the Secretary, and the burden of proof shall be on the Secretary with respect to such value.

(2)Cancellation
If, in the case of property subject to a restriction which by its terms will never lapse, the restriction is canceled, then, unless the taxpayer establishes—
(A)that such cancellation was not compensatory, and
(B)that the person, if any, who would be allowed a deduction if the cancellation were treated as compensatory, will treat the transaction as not compensatory, as evidenced in such manner as the Secretary shall prescribe by regulations,
the excess of the fair market value of the property (computed without regard to the restrictions) at the time of cancellation over the sum of—
(C)the fair market value of such property (computed by taking the restriction into account) immediately before the cancellation, and
(D)the amount, if any, paid for the cancellation,
shall be treated as compensation for the taxable year in which such cancellation occurs.
(e)Applicability of section
This section shall not apply to—
(1)a transaction to which section 421 applies,
(2)a transfer to or from a trust described in section 401(a) or a transfer under an annuity plan which meets the requirements of section 404(a)(2),
(3)the transfer of an option without a readily ascertainable fair market value,
(4)the transfer of property pursuant to the exercise of an option with a readily ascertainable fair market value at the date of grant, or
(5)group-term life insurance to which section 79 applies.
(f)Holding period
In determining the period for which the taxpayer has held property to which subsection (a) applies, there shall be included only the period beginning at the first time his rights in such property are transferable or are not subject to a substantial risk of forfeiture, whichever occurs earlier.

(g)Certain exchanges
If property to which subsection (a) applies is exchanged for property subject to restrictions and conditions substantially similar to those to which the property given in such exchange was subject, and if section 354, 355, 356, or 1036 (or so much of section 1031 as relates to section 1036) applied to such exchange, or if such exchange was pursuant to the exercise of a conversion privilege—
(1)such exchange shall be disregarded for purposes of subsection (a), and
(2)the property received shall be treated as property to which subsection (a) applies.
(h)Deduction by employer
In the case of a transfer of property to which this section applies or a cancellation of a restriction described in subsection (d), there shall be allowed as a deduction under section 162, to the person for whom were performed the services in connection with which such property was transferred, an amount equal to the amount included under subsection (a), (b), or (d)(2) in the gross income of the person who performed such services. Such deduction shall be allowed for the taxable year of such person in which or with which ends the taxable year in which such amount is included in the gross income of the person who performed such services.

(i)Qualified equity grants
(1)In general
For purposes of this subtitle—
(A)Timing of inclusion
If qualified stock is transferred to a qualified employee who makes an election with respect to such stock under this subsection, subsection (a) shall be applied by including the amount determined under such subsection with respect to such stock in income of the employee in the taxable year determined under subparagraph (B) in lieu of the taxable year described in subsection (a).

(B)Taxable year determined
The taxable year determined under this subparagraph is the taxable year of the employee which includes the earliest of—
(i)the first date such qualified stock becomes transferable (including, solely for purposes of this clause, becoming transferable to the employer),
(ii)the date the employee first becomes an excluded employee,
(iii)the first date on which any stock of the corporation which issued the qualified stock becomes readily tradable on an established securities market (as determined by the Secretary, but not including any market unless such market is recognized as an established securities market by the Secretary for purposes of a provision of this title other than this subsection),
(iv)the date that is 5 years after the first date the rights of the employee in such stock are transferable or are not subject to a substantial risk of forfeiture, whichever occurs earlier, or
(v)the date on which the employee revokes (at such time and in such manner as the Secretary provides) the election under this subsection with respect to such stock.
(2)Qualified stock
(A)In general
For purposes of this subsection, the term "qualified stock" means, with respect to any qualified employee, any stock in a corporation which is the employer of such employee, if—
(i)such stock is received—
(I)in connection with the exercise of an option, or
(II)in settlement of a restricted stock unit, and
(ii)such option or restricted stock unit was granted by the corporation—
(I)in connection with the performance of services as an employee, and
(II)during a calendar year in which such corporation was an eligible corporation.
(B)Limitation
The term "qualified stock" shall not include any stock if the employee may sell such stock to, or otherwise receive cash in lieu of stock from, the corporation at the time that the rights of the employee in such stock first become transferable or not subject to a substantial risk of forfeiture.

(C)Eligible corporation
For purposes of subparagraph (A)(ii)(II)—
(i)In general
The term "eligible corporation" means, with respect to any calendar year, any corporation if—
(I)no stock of such corporation (or any predecessor of such corporation) is readily tradable on an established securities market (as determined under paragraph (1)(B)(iii)) during any preceding calendar year, and
(II)such corporation has a written plan under which, in such calendar year, not less than 80 percent of all employees who provide services to such corporation in the United States (or any possession of the United States) are granted stock options, or are granted restricted stock units, with the same rights and privileges to receive qualified stock.
(ii)Same rights and privileges
For purposes of clause (i)(II)—
(I)except as provided in subclauses (II) and (III), the determination of rights and privileges with respect to stock shall be made in a similar manner as under section 423(b)(5),
(II)employees shall not fail to be treated as having the same rights and privileges to receive qualified stock solely because the number of shares available to all employees is not equal in amount, so long as the number of shares available to each employee is more than a de minimis amount, and
(III)rights and privileges with respect to the exercise of an option shall not be treated as the same as rights and privileges with respect to the settlement of a restricted stock unit.
(iii)Employee
For purposes of clause (i)(II), the term "employee" shall not include any employee described in section 4980E(d)(4) or any excluded employee.

(iv)Special rule for calendar years before 2018
In the case of any calendar year beginning before January 1, 2018, clause (i)(II) shall be applied without regard to whether the rights and privileges with respect to the qualified stock are the same.

(3)Qualified employee; excluded employee
For purposes of this subsection—
(A)In general
The term "qualified employee" means any individual who—
(i)is not an excluded employee, and
(ii)agrees in the election made under this subsection to meet such requirements as are determined by the Secretary to be necessary to ensure that the withholding requirements of the corporation under chapter 24 with respect to the qualified stock are met.
(B)Excluded employee
The term "excluded employee" means, with respect to any corporation, any individual—
(i)who is a 1-percent owner (within the meaning of section 416(i)(1)(B)(ii)) at any time during the calendar year or who was such a 1 percent owner at any time during the 10 preceding calendar years,
(ii)who is or has been at any prior time—
(I)the chief executive officer of such corporation or an individual acting in such a capacity, or
(II)the chief financial officer of such corporation or an individual acting in such a capacity,
(iii)who bears a relationship described in section 318(a)(1) to any individual described in subclause (I) or (II) of clause (ii), or
(iv)who is one of the 4 highest compensated officers of such corporation for the taxable year, or was one of the 4 highest compensated officers of such corporation for any of the 10 preceding taxable years, determined with respect to each such taxable year on the basis of the shareholder disclosure rules for compensation under the Securities Exchange Act of 1934 (as if such rules applied to such corporation).
(4)Election
(A)Time for making election
An election with respect to qualified stock shall be made under this subsection no later than 30 days after the first date the rights of the employee in such stock are transferable or are not subject to a substantial risk of forfeiture, whichever occurs earlier, and shall be made in a manner similar to the manner in which an election is made under subsection (b).

(B)Limitations
No election may be made under this section with respect to any qualified stock if—
(i)the qualified employee has made an election under subsection (b) with respect to such qualified stock,
(ii)any stock of the corporation which issued the qualified stock is readily tradable on an established securities market (as determined under paragraph (1)(B)(iii)) at any time before the election is made, or
(iii)such corporation purchased any of its outstanding stock in the calendar year preceding the calendar year which includes the first date the rights of the employee in such stock are transferable or are not subject to a substantial risk of forfeiture, unless—
(I)not less than 25 percent of the total dollar amount of the stock so purchased is deferral stock, and
(II)the determination of which individuals from whom deferral stock is purchased is made on a reasonable basis.
(C)Definitions and special rules related to limitation on stock redemptions
(i)Deferral stock
For purposes of this paragraph, the term "deferral stock" means stock with respect to which an election is in effect under this subsection.

(ii)Deferral stock with respect to any individual not taken into account if individual holds deferral stock with longer deferral period
Stock purchased by a corporation from any individual shall not be treated as deferral stock for purposes of subparagraph (B)(iii) if such individual (immediately after such purchase) holds any deferral stock with respect to which an election has been in effect under this subsection for a longer period than the election with respect to the stock so purchased.

(iii)Purchase of all outstanding deferral stock
The requirements of subclauses (I) and (II) of subparagraph (B)(iii) shall be treated as met if the stock so purchased includes all of the corporation's outstanding deferral stock.

(iv)Reporting
Any corporation which has outstanding deferral stock as of the beginning of any calendar year and which purchases any of its outstanding stock during such calendar year shall include on its return of tax for the taxable year in which, or with which, such calendar year ends the total dollar amount of its outstanding stock so purchased during such calendar year and such other information as the Secretary requires for purposes of administering this paragraph.

(5)Controlled groups
For purposes of this subsection, all persons treated as a single employer under section 414(b) shall be treated as 1 corporation.

(6)Notice requirement
Any corporation which transfers qualified stock to a qualified employee shall, at the time that (or a reasonable period before) an amount attributable to such stock would (but for this subsection) first be includible in the gross income of such employee—
(A)certify to such employee that such stock is qualified stock, and
(B)notify such employee—
(i)that the employee may be eligible to elect to defer income on such stock under this subsection, and
(ii)that, if the employee makes such an election—
(I)the amount of income recognized at the end of the deferral period will be based on the value of the stock at the time at which the rights of the employee in such stock first become transferable or not subject to substantial risk of forfeiture, notwithstanding whether the value of the stock has declined during the deferral period,
(II)the amount of such income recognized at the end of the deferral period will be subject to withholding under section 3401(i) at the rate determined under section 3402(t), and
(III)the responsibilities of the employee (as determined by the Secretary under paragraph (3)(A)(ii)) with respect to such withholding.
(7)Restricted stock units
This section (other than this subsection), including any election under subsection (b), shall not apply to restricted stock units.
"""



######################################## ELEMENTOS DO PROMPT ########################################

##### Elemento do prompt 1: role `user`
# Certifique-se de que sua chamada à API de Mensagens sempre comece com um role `user` no array de mensagens.
# A função get_completion() conforme definida acima fará isso automaticamente para você.

##### Elemento do prompt 2: Contexto da tarefa
# Forneça ao Claude contexto sobre o papel que ele deve assumir ou quais metas e tarefas gerais você quer que ele execute com o prompt.
# É melhor colocar o contexto no início do corpo do prompt.
TASK_CONTEXT = ""

##### Elemento do prompt 3: Contexto de tom
# Se importante para a interação, diga ao Claude qual tom ele deve usar.
# Este elemento pode não ser necessário dependendo da tarefa.
TONE_CONTEXT = ""

##### Elemento do prompt 4: Dados de entrada para processar
# Se houver dados que o Claude precise processar no prompt, inclua-os aqui dentro de tags XML relevantes.
# Fique à vontade para incluir múltiplos dados, mas certifique-se de colocar cada um em seu próprio conjunto de tags XML.
# Este elemento pode não ser necessário dependendo da tarefa. A ordem também é flexível.
INPUT_DATA = ""

##### Elemento do prompt 5: Exemplos
# Forneça ao Claude pelo menos um exemplo de resposta ideal que ele possa emular. Coloque dentro de tags XML <example></example>. Fique à vontade para fornecer múltiplos exemplos.
# Se você fornecer múltiplos exemplos, dê ao Claude contexto sobre o que é um exemplo, e coloque cada exemplo em seu próprio conjunto de tags XML.
# Exemplos são provavelmente a ferramenta mais eficaz em trabalho de conhecimento para fazer o Claude se comportar como desejado.
# Certifique-se de dar ao Claude exemplos de casos extremos comuns. Se seu prompt usa um rascunho, é eficaz dar exemplos de como o rascunho deve parecer.
# Em geral, mais exemplos = melhor.
EXAMPLES = ""

##### Elemento do prompt 6: Descrição detalhada da tarefa e regras
# Expanda sobre as tarefas específicas que você quer que o Claude faça, bem como quaisquer regras que o Claude possa ter que seguir.
# É aqui também que você pode dar ao Claude uma "saída" se ele não tiver uma resposta ou não souber.
# É ideal mostrar esta descrição e regras a um amigo para garantir que estejam dispostas de forma lógica e que quaisquer palavras ambíguas estejam claramente definidas.
TASK_DESCRIPTION = ""

##### Elemento do prompt 7: Descrição ou solicitação imediata da tarefa #####
# "Lembre" ao Claude ou diga ao Claude exatamente o que ele deve fazer imediatamente para cumprir a tarefa do prompt.
# É aqui também que você colocaria variáveis adicionais como a pergunta do usuário.
# Geralmente não há problema em reiterar ao Claude sua tarefa imediata. É melhor fazer isso no final de um prompt longo.
# Isso produzirá melhores resultados do que colocar isso no início.
# Também é geralmente boa prática colocar a consulta do usuário próxima ao final do prompt.
IMMEDIATE_TASK = ""

##### Elemento do prompt 8: Pré-cognição (pensar passo a passo)
# Para tarefas com múltiplas etapas, é bom dizer ao Claude para pensar passo a passo antes de dar uma resposta
# Às vezes, você pode até ter que dizer "Antes de dar sua resposta..." apenas para garantir que o Claude faça isso primeiro.
# Não é necessário em todos os prompts, mas se incluído, é melhor fazer isso no final de um prompt longo e logo após a solicitação ou descrição final da tarefa imediata.
PRECOGNITION = ""

##### Elemento do prompt 9: Formatação de saída
# Se houver uma maneira específica de formatar a resposta do Claude, diga claramente ao Claude qual é esse formato.
# Este elemento pode não ser necessário dependendo da tarefa.
# Se você incluí-lo, colocá-lo no final do prompt é melhor do que no início.
OUTPUT_FORMATTING = ""

##### Elemento do prompt 10: Preenchimento da resposta do Claude (se houver)
# Um espaço para iniciar a resposta do Claude com algumas palavras pré-preenchidas para orientar o comportamento ou a resposta do Claude.
# Se você quiser pré-preencher a resposta do Claude, deve colocar isso no role `assistant` na chamada da API.
# Este elemento pode não ser necessário dependendo da tarefa.
PREFILL = ""



######################################## COMBINAR ELEMENTOS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Imprime o prompt completo
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print("TURNO DO USUÁRIO")
print(PROMPT)
print("\nTURNO DO ASSISTENTE")
print(PREFILL)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

❓ Se quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_9_1_solution; print(exercise_9_1_solution)

### Exercício 9.2 - Codebot
Neste exercício, escreveremos um prompt para um **bot de assistência e ensino de programação que lê código e oferece correções orientadas quando apropriado**. Preencha os campos dos elementos do prompt com conteúdo que corresponda à descrição e aos exemplos que você viu nos exemplos anteriores de prompts complexos. Depois de preencher todos os elementos do prompt que deseja preencher, execute a célula para ver o prompt concatenado e a resposta do Claude.

Sugerimos que você leia o conteúdo da variável (neste caso, `{CODE}`) para entender com qual conteúdo o Claude deve trabalhar. Certifique-se de referenciar `{CODE}` diretamente no seu prompt em algum lugar (usando sintaxe f-string como nos outros exemplos) para que o conteúdo real da variável possa ser substituído.

In [ ]:
######################################## VARIÁVEIS DE ENTRADA ########################################

# Variável de entrada - o código que o Claude precisa ler e ajudar o usuário a corrigir
CODE = """
# Function to print multiplicative inverses
def print_multiplicative_inverses(x, n):
  for i in range(n):
    print(x / i) 
"""



######################################## ELEMENTOS DO PROMPT ########################################

##### Elemento do prompt 1: role `user`
# Certifique-se de que sua chamada à API de Mensagens sempre comece com um role `user` no array de mensagens.
# A função get_completion() conforme definida acima fará isso automaticamente para você.

##### Elemento do prompt 2: Contexto da tarefa
# Forneça ao Claude contexto sobre o papel que ele deve assumir ou quais metas e tarefas gerais você quer que ele execute com o prompt.
# É melhor colocar o contexto no início do corpo do prompt.
TASK_CONTEXT = ""

##### Elemento do prompt 3: Contexto de tom
# Se importante para a interação, diga ao Claude qual tom ele deve usar.
# Este elemento pode não ser necessário dependendo da tarefa.
TONE_CONTEXT = ""

##### Elemento do prompt 4: Descrição detalhada da tarefa e regras
# Expanda sobre as tarefas específicas que você quer que o Claude faça, bem como quaisquer regras que o Claude possa ter que seguir.
# É aqui também que você pode dar ao Claude uma "saída" se ele não tiver uma resposta ou não souber.
# É ideal mostrar esta descrição e regras a um amigo para garantir que estejam dispostas de forma lógica e que quaisquer palavras ambíguas estejam claramente definidas.
TASK_DESCRIPTION = ""

##### Elemento do prompt 5: Exemplos
# Forneça ao Claude pelo menos um exemplo de resposta ideal que ele possa emular. Coloque dentro de tags XML <example></example>. Fique à vontade para fornecer múltiplos exemplos.
# Se você fornecer múltiplos exemplos, dê ao Claude contexto sobre o que é um exemplo, e coloque cada exemplo em seu próprio conjunto de tags XML.
# Exemplos são provavelmente a ferramenta mais eficaz em trabalho de conhecimento para fazer o Claude se comportar como desejado.
# Certifique-se de dar ao Claude exemplos de casos extremos comuns. Se seu prompt usa um rascunho, é eficaz dar exemplos de como o rascunho deve parecer.
# Em geral, mais exemplos = melhor.
EXAMPLES = ""

##### Elemento do prompt 6: Dados de entrada para processar
# Se houver dados que o Claude precise processar no prompt, inclua-os aqui dentro de tags XML relevantes.
# Fique à vontade para incluir múltiplos dados, mas certifique-se de colocar cada um em seu próprio conjunto de tags XML.
# Este elemento pode não ser necessário dependendo da tarefa. A ordem também é flexível.
INPUT_DATA = ""

##### Elemento do prompt 7: Descrição ou solicitação imediata da tarefa #####
# "Lembre" ao Claude ou diga ao Claude exatamente o que ele deve fazer imediatamente para cumprir a tarefa do prompt.
# É aqui também que você colocaria variáveis adicionais como a pergunta do usuário.
# Geralmente não há problema em reiterar ao Claude sua tarefa imediata. É melhor fazer isso no final de um prompt longo.
# Isso produzirá melhores resultados do que colocar isso no início.
# Também é geralmente boa prática colocar a consulta do usuário próxima ao final do prompt.
IMMEDIATE_TASK = ""

##### Elemento do prompt 8: Pré-cognição (pensar passo a passo)
# Para tarefas com múltiplas etapas, é bom dizer ao Claude para pensar passo a passo antes de dar uma resposta
# Às vezes, você pode até ter que dizer "Antes de dar sua resposta..." apenas para garantir que o Claude faça isso primeiro.
# Não é necessário em todos os prompts, mas se incluído, é melhor fazer isso no final de um prompt longo e logo após a solicitação ou descrição final da tarefa imediata.
PRECOGNITION = ""

##### Elemento do prompt 9: Formatação de saída
# Se houver uma maneira específica de formatar a resposta do Claude, diga claramente ao Claude qual é esse formato.
# Este elemento pode não ser necessário dependendo da tarefa.
# Se você incluí-lo, colocá-lo no final do prompt é melhor do que no início.
OUTPUT_FORMATTING = ""

##### Elemento do prompt 10: Preenchimento da resposta do Claude (se houver)
# Um espaço para iniciar a resposta do Claude com algumas palavras pré-preenchidas para orientar o comportamento ou a resposta do Claude.
# Se você quiser pré-preencher a resposta do Claude, deve colocar isso no role `assistant` na chamada da API.
# Este elemento pode não ser necessário dependendo da tarefa.
PREFILL = ""



######################################## COMBINAR ELEMENTOS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Imprime o prompt completo
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print("TURNO DO USUÁRIO")
print(PROMPT)
print("\nTURNO DO ASSISTENTE")
print(PREFILL)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

❓ Se quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_9_2_solution; print(exercise_9_2_solution)

---

## Parabéns e Próximos Passos!

Se você completou todos os exercícios, **você está agora no top 0,1% dos sussurradores de LLM**. Um dos mais seletos!

As técnicas que você aprendeu, desde pensar passo a passo até atribuir papéis, usar exemplos e escrever de forma clara em geral, podem ser **mescladas, remixadas e adaptadas de inúmeras maneiras**.

Engenharia de prompt é uma disciplina muito nova, então mantenha a mente aberta. Você pode ser aquele que descobre o próximo grande truque de prompting.

Se você quiser ver **mais exemplos de bons prompts** para inspiração:					
- Aprenda com exemplos de prompts prontos para produção do nosso [cookbook](https://anthropic.com/cookbook)
- Leia nosso [guia de prompting](https://docs.anthropic.com/claude/docs/prompt-engineering)
- Confira nossa [biblioteca de prompts](https://anthropic.com/prompts) para inspiração
- Experimente nosso [metaprompt](https://docs.anthropic.com/claude/docs/helper-metaprompt-experimental) experimental para fazer o Claude escrever templates de prompt para você!
- Faça perguntas em nosso [servidor discord](https://anthropic.com/discord)
- Aprenda sobre os [parâmetros da API da Anthropic](https://docs.anthropic.com/claude/reference/complete_post) como temperature e `max_tokens`
- Se você se sentir acadêmico, leia alguns [artigos](https://www.promptingguide.ai/papers) sobre engenharia de prompt
- Pratique construindo prompts para fazer o Claude fazer algo que te interessa

Se você quiser aprender sobre técnicas de prompting verdadeiramente avançadas além do escopo deste tutorial, clique para acessar o apêndice! Mas primeiro, execute a célula abaixo.

In [ ]:
# Prompt
PROMPT = "Write an ode to a fabulous student who has just completed a course on prompt engineering, in the form of a sonnet."

# Imprime a resposta do Claude
print(get_completion(PROMPT))

---

## Playground de Exemplos

Esta é uma área para você experimentar livremente com os exemplos de prompt mostrados nesta lição e ajustar prompts para ver como isso pode afetar as respostas do Claude.

In [ ]:
######################################## VARIÁVEIS DE ENTRADA ########################################

# Primeira variável de entrada - o histórico da conversa (também pode ser adicionado como mensagens anteriores `user` e `assistant` na chamada da API)
HISTORY = """Customer: Give me two possible careers for sociology majors.

Joe: Here are two potential careers for sociology majors:

- Social worker - Sociology provides a strong foundation for understanding human behavior and social systems. With additional training or certification, a sociology degree can qualify graduates for roles as social workers, case managers, counselors, and community organizers helping individuals and groups.

- Human resources specialist - An understanding of group dynamics and organizational behavior from sociology is applicable to careers in human resources. Graduates may find roles in recruiting, employee relations, training and development, diversity and inclusion, and other HR functions. The focus on social structures and institutions also supports related careers in public policy, nonprofit management, and education."""

# Segunda variável de entrada - a pergunta do usuário
QUESTION = "Which of the two careers requires more than a Bachelor's degree?"



######################################## ELEMENTOS DO PROMPT ########################################

##### Elemento do prompt 1: role `user`
# Certifique-se de que sua chamada à API de Mensagens sempre comece com um role `user` no array de mensagens.
# A função get_completion() conforme definida acima fará isso automaticamente para você.

##### Elemento do prompt 2: Contexto da tarefa
# Forneça ao Claude contexto sobre o papel que ele deve assumir ou quais metas e tarefas gerais você quer que ele execute com o prompt.
# É melhor colocar o contexto no início do corpo do prompt.
TASK_CONTEXT = "You will be acting as an AI career coach named Joe created by the company AdAstra Careers. Your goal is to give career advice to users. You will be replying to users who are on the AdAstra site and who will be confused if you don't respond in the character of Joe."

##### Elemento do prompt 3: Contexto de tom
# Se importante para a interação, diga ao Claude qual tom ele deve usar.
# Este elemento pode não ser necessário dependendo da tarefa.
TONE_CONTEXT = "You should maintain a friendly customer service tone."

##### Elemento do prompt 4: Descrição detalhada da tarefa e regras
# Expanda sobre as tarefas específicas que você quer que o Claude faça, bem como quaisquer regras que o Claude possa ter que seguir.
# É aqui também que você pode dar ao Claude uma "saída" se ele não tiver uma resposta ou não souber.
# É ideal mostrar esta descrição e regras a um amigo para garantir que estejam dispostas de forma lógica e que quaisquer palavras ambíguas estejam claramente definidas.
TASK_DESCRIPTION = """Here are some important rules for the interaction:
- Always stay in character, as Joe, an AI from AdAstra Careers
- If you are unsure how to respond, say \"Sorry, I didn't understand that. Could you rephrase your question?\"
- If someone asks something irrelevant, say, \"Sorry, I am Joe and I give career advice. Do you have a career question today I can help you with?\""""

##### Elemento do prompt 5: Exemplos
# Forneça ao Claude pelo menos um exemplo de resposta ideal que ele possa emular. Coloque dentro de tags XML <example></example>. Fique à vontade para fornecer múltiplos exemplos.
# Se você fornecer múltiplos exemplos, dê ao Claude contexto sobre o que é um exemplo, e coloque cada exemplo em seu próprio conjunto de tags XML.
# Exemplos são provavelmente a ferramenta mais eficaz em trabalho de conhecimento para fazer o Claude se comportar como desejado.
# Certifique-se de dar ao Claude exemplos de casos extremos comuns. Se seu prompt usa um rascunho, é eficaz dar exemplos de como o rascunho deve parecer.
# Em geral, mais exemplos = melhor.
EXAMPLES = """Here is an example of how to respond in a standard interaction:
<example>
Customer: Hi, how were you created and what do you do?
Joe: Hello! My name is Joe, and I was created by AdAstra Careers to give career advice. What can I help you with today?
</example>"""

##### Elemento do prompt 6: Dados de entrada para processar
# Se houver dados que o Claude precise processar no prompt, inclua-os aqui dentro de tags XML relevantes.
# Fique à vontade para incluir múltiplos dados, mas certifique-se de colocar cada um em seu próprio conjunto de tags XML.
# Este elemento pode não ser necessário dependendo da tarefa. A ordem também é flexível.
INPUT_DATA = f"""Here is the conversational history (between the user and you) prior to the question. It could be empty if there is no history:
<history>
{HISTORY}
</history>

Here is the user's question:
<question>
{QUESTION}
</question>"""

##### Elemento do prompt 7: Descrição ou solicitação imediata da tarefa #####
# "Lembre" ao Claude ou diga ao Claude exatamente o que ele deve fazer imediatamente para cumprir a tarefa do prompt.
# É aqui também que você colocaria variáveis adicionais como a pergunta do usuário.
# Geralmente não há problema em reiterar ao Claude sua tarefa imediata. É melhor fazer isso no final de um prompt longo.
# Isso produzirá melhores resultados do que colocar isso no início.
# Também é geralmente boa prática colocar a consulta do usuário próxima ao final do prompt.
IMMEDIATE_TASK = "How do you respond to the user's question?"

##### Elemento do prompt 8: Pré-cognição (pensar passo a passo)
# Para tarefas com múltiplas etapas, é bom dizer ao Claude para pensar passo a passo antes de dar uma resposta
# Às vezes, você pode até ter que dizer "Antes de dar sua resposta..." apenas para garantir que o Claude faça isso primeiro.
# Não é necessário em todos os prompts, mas se incluído, é melhor fazer isso no final de um prompt longo e logo após a solicitação ou descrição final da tarefa imediata.
PRECOGNITION = "Think about your answer first before you respond."

##### Elemento do prompt 9: Formatação de saída
# Se houver uma maneira específica de formatar a resposta do Claude, diga claramente ao Claude qual é esse formato.
# Este elemento pode não ser necessário dependendo da tarefa.
# Se você incluí-lo, colocá-lo no final do prompt é melhor do que no início.
OUTPUT_FORMATTING = "Put your response in <response></response> tags."

##### Elemento do prompt 10: Preenchimento da resposta do Claude (se houver)
# Um espaço para iniciar a resposta do Claude com algumas palavras pré-preenchidas para orientar o comportamento ou a resposta do Claude.
# Se você quiser pré-preencher a resposta do Claude, deve colocar isso no role `assistant` na chamada da API.
# Este elemento pode não ser necessário dependendo da tarefa.
PREFILL = "[Joe] <response>"



######################################## COMBINAR ELEMENTOS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Imprime o prompt completo
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print("TURNO DO USUÁRIO")
print(PROMPT)
print("\nTURNO DO ASSISTENTE")
print(PREFILL)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
######################################## VARIÁVEIS DE ENTRADA ########################################

# Primeira variável de entrada - o documento legal
LEGAL_RESEARCH = """<search_results>
<search_result id=1>
The animal health industry became caught up in a number of patent and trademark lawsuits during the past year. In 1994, Barclay Slocum obtained patents for the tibial plateau leveling osteotomy procedure, which is used in the treatment of dogs with cranial cruciate ligament rupture, and for the devices used in the procedure. During 2006, Slocum Enterprises filed a patent infringement suit against New Generation Devices, arguing that the Unity Cruciate Plate manufactured by New Generation infringed on the patent for the Slocum TPLO plate. However, the court never reached a decision on the issue of patent infringement, ruling that it did not have jurisdiction on the basis of the small number of plates sold in the state in which the case was filed and the information provided on a Web site maintained by Slocum Enterprises. Other patent battles waged during 2006 concerned the use of laser technology for onychectomy in cats, pet identification chips, pig vaccines, and pet "deshedding" tools.
</search_result>
<search_result id=2>
In Canada, the British Columbia Veterinary Medical Association brought suit against a nonveterinarian, claiming that he engaged in cutting or otherwise removing hooks from horses' teeth and floating horses' teeth with power and manual tools, provided advice and diagnoses in return for a fee, and held himself out as being qualified and willing to provide treatment with respect to these activities. The court held that the intention of the legislature in passing the Veterinary Profession Act was the protection of the public and animals and further held that monopolistic statutes serve the purpose of protecting the public. In addition, the court concluded that dentistry, at its core, relates to the health of the teeth and gums; is distinct from cosmetic and other types of care of animals; and, therefore, falls under the definition of the practice of veterinary medicine. The nonveterinarian was enjoined from providing services without a veterinarian supervising the procedures.
</search_result>
<search_result id=3>
The aftermath of Hurricane Katrina, which hit the Gulf Coast of the United States during 2005, spurred changes to the way animals are treated during natural disasters. In 2006, Hawaii, Louisiana, and New Hampshire all enacted laws that address issues regarding the care of animals during disasters, such as providing shelters for pets and allowing service animals to be kept with the people they serve. In addition, Congress passed, and the President signed, the Pet Evacuation and Transportation Standards Act during 2006, which requires state and local emergency preparedness authorities to include in their evacuation plans information on how they will accommodate household pets and service animals in case of a disaster. California passed a law that will require its Office of Emergency Services, Department of Agriculture, and other agencies involved with disaster response preparation to develop a plan for the needs of service animals, livestock, equids, and household pets in the event of a disaster or major emergency.
</search_result>
</search_results>"""

# Segunda variável de entrada - a pergunta do usuário
QUESTION = "Are there any laws about what to do with pets during a hurricane?"



######################################## ELEMENTOS DO PROMPT ########################################

##### Elemento do prompt 1: role `user`
# Certifique-se de que sua chamada à API de Mensagens sempre comece com um role `user` no array de mensagens.
# A função get_completion() conforme definida acima fará isso automaticamente para você.

##### Elemento do prompt 2: Contexto da tarefa
# Forneça ao Claude contexto sobre o papel que ele deve assumir ou quais metas e tarefas gerais você quer que ele execute com o prompt.
# É melhor colocar o contexto no início do corpo do prompt.
TASK_CONTEXT = "You are an expert lawyer."

##### Elemento do prompt 3: Contexto de tom
# Se importante para a interação, diga ao Claude qual tom ele deve usar.
# Este elemento pode não ser necessário dependendo da tarefa.
TONE_CONTEXT = ""

##### Elemento do prompt 4: Dados de entrada para processar
# Se houver dados que o Claude precise processar no prompt, inclua-os aqui dentro de tags XML relevantes.
# Fique à vontade para incluir múltiplos dados, mas certifique-se de colocar cada um em seu próprio conjunto de tags XML.
# Este elemento pode não ser necessário dependendo da tarefa. A ordem também é flexível.
INPUT_DATA = f"""Here is some research that's been compiled. Use it to answer a legal question from the user.
<legal_research>
{LEGAL_RESEARCH}
</legal_research>"""

##### Elemento do prompt 5: Exemplos
# Forneça ao Claude pelo menos um exemplo de resposta ideal que ele possa emular. Coloque dentro de tags XML <example></example>. Fique à vontade para fornecer múltiplos exemplos.
# Se você fornecer múltiplos exemplos, dê ao Claude contexto sobre o que é um exemplo, e coloque cada exemplo em seu próprio conjunto de tags XML.
# Exemplos são provavelmente a ferramenta mais eficaz em trabalho de conhecimento para fazer o Claude se comportar como desejado.
# Certifique-se de dar ao Claude exemplos de casos extremos comuns. Se seu prompt usa um rascunho, é eficaz dar exemplos de como o rascunho deve parecer.
# Em geral, mais exemplos = melhor.
EXAMPLES = """When citing the legal research in your answer, please use brackets containing the search index ID, followed by a period. Put these at the end of the sentence that's doing the citing. Examples of proper citation format:

<examples>
<example>
The statute of limitations expires after 10 years for crimes like this. [3].
</example>
<example>
However, the protection does not apply when it has been specifically waived by both parties. [5].
</example>
</examples>"""

##### Elemento do prompt 6: Descrição detalhada da tarefa e regras
# Expanda sobre as tarefas específicas que você quer que o Claude faça, bem como quaisquer regras que o Claude possa ter que seguir.
# É aqui também que você pode dar ao Claude uma "saída" se ele não tiver uma resposta ou não souber.
# É ideal mostrar esta descrição e regras a um amigo para garantir que estejam dispostas de forma lógica e que quaisquer palavras ambíguas estejam claramente definidas.
TASK_DESCRIPTION = """Write a clear, concise answer to this question:

<question>
{QUESTION}
</question>

It should be no more than a couple of paragraphs. If possible, it should conclude with a single sentence directly answering the user's question. However, if there is not sufficient information in the compiled research to produce such an answer, you may demur and write \"Sorry, I do not have sufficient information at hand to answer this question.\"."""

##### Elemento do prompt 7: Descrição ou solicitação imediata da tarefa #####
# "Lembre" ao Claude ou diga ao Claude exatamente o que ele deve fazer imediatamente para cumprir a tarefa do prompt.
# É aqui também que você colocaria variáveis adicionais como a pergunta do usuário.
# Geralmente não há problema em reiterar ao Claude sua tarefa imediata. É melhor fazer isso no final de um prompt longo.
# Isso produzirá melhores resultados do que colocar isso no início.
# Também é geralmente boa prática colocar a consulta do usuário próxima ao final do prompt.
IMMEDIATE_TASK = ""

##### Elemento do prompt 8: Pré-cognição (pensar passo a passo)
# Para tarefas com múltiplas etapas, é bom dizer ao Claude para pensar passo a passo antes de dar uma resposta
# Às vezes, você pode até ter que dizer "Antes de dar sua resposta..." apenas para garantir que o Claude faça isso primeiro.
# Não é necessário em todos os prompts, mas se incluído, é melhor fazer isso no final de um prompt longo e logo após a solicitação ou descrição final da tarefa imediata.
PRECOGNITION = "Before you answer, pull out the most relevant quotes from the research in <relevant_quotes> tags."

##### Elemento do prompt 9: Formatação de saída
# Se houver uma maneira específica de formatar a resposta do Claude, diga claramente ao Claude qual é esse formato.
# Este elemento pode não ser necessário dependendo da tarefa.
# Se você incluí-lo, colocá-lo no final do prompt é melhor do que no início.
OUTPUT_FORMATTING = "Put your two-paragraph response in <answer> tags."

##### Elemento do prompt 10: Preenchimento da resposta do Claude (se houver)
# Um espaço para iniciar a resposta do Claude com algumas palavras pré-preenchidas para orientar o comportamento ou a resposta do Claude.
# Se você quiser pré-preencher a resposta do Claude, deve colocar isso no role `assistant` na chamada da API.
# Este elemento pode não ser necessário dependendo da tarefa.
PREFILL = "<relevant_quotes>"



######################################## COMBINAR ELEMENTOS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Imprime o prompt completo
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print("TURNO DO USUÁRIO")
print(PROMPT)
print("\nTURNO DO ASSISTENTE")
print(PREFILL)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))